In [1]:
!pip install biopython pydssp --quiet

### Step 1: 3D Structure Prediction (Swiss-Model) + Tertiary Identification (Foldseek)

In [2]:
import requests
import time
import os

def fetch_alphafold_structure(uniprot_id):
    print(f"--- STEP 1A: Fetching 3D Structure (AlphaFold DB for {uniprot_id}) ---")
    url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"

    print("Querying AlphaFold Database (Instant lookup)...")
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"AlphaFold DB lookup failed! HTTP {response.status_code}. Ensure the UniProt ID is valid.")

    data = response.json()
    if not data:
        raise Exception("No AlphaFold structure found for this UniProt ID.")

    pdb_url = data[0].get("pdbUrl")

    pdb_response = requests.get(pdb_url)
    if pdb_response.status_code != 200:
        raise Exception("Failed to download the physical PDB coordinates.")

    pdb_data = pdb_response.text
    pdb_filename = f"alphafold_{uniprot_id}.pdb"

    with open(pdb_filename, "w") as f:
        f.write(pdb_data)

    print(f"✅ 3D Structure downloaded instantly and saved as {pdb_filename}")
    return pdb_filename, pdb_data

def identify_structure(pdb_filename, pdb_data):
    print("\n--- STEP 1B: Tertiary Identification (Foldseek) ---")
    fs_submit_url = "https://search.foldseek.com/api/ticket"
    files = {'q': (pdb_filename, pdb_data)}
    data = [('mode', '3diaa'), ('database[]', 'pdb100'), ('database[]', 'afdb50')]

    res = requests.post(fs_submit_url, files=files, data=data)
    if res.status_code != 200:
        raise Exception(f"Foldseek submission rejected! HTTP {res.status_code}: {res.text}")

    ticket_id = res.json().get('id')
    status_url = f"https://search.foldseek.com/api/ticket/{ticket_id}"

    while True:
        status_res = requests.get(status_url).json()
        status = status_res.get('status')

        if status == 'COMPLETE':
            break
        elif status in ['ERROR', 'FAILED']:
            print("\n❌ FOLDSEEK ERROR: Foldseek rejected the PDB structure.")
            raise Exception("Foldseek rejected the AlphaFold output.")

        time.sleep(2)

    result_url = f"https://search.foldseek.com/api/result/{ticket_id}/0"
    raw_results = requests.get(result_url).json()

    top_hits = {}

    if isinstance(raw_results, dict) and 'results' in raw_results:
        raw_results = raw_results['results']
    if not isinstance(raw_results, list):
        raw_results = [raw_results]

    for db_block in raw_results:
        if isinstance(db_block, dict) and 'alignments' in db_block:
            db_name = db_block.get('target', db_block.get('db', 'Unknown_DB'))
            alignments_array = db_block['alignments']

            if isinstance(alignments_array, list) and len(alignments_array) > 0:
                query_hits = alignments_array[0]

                if isinstance(query_hits, list) and len(query_hits) > 0:
                    top_hit = query_hits[0]
                    top_hits[db_name] = top_hit

                    target_name = top_hit.get('target', 'Unknown')
                    e_value = top_hit.get('eval', 'N/A')
                    print(f"✅ Top match in {db_name}: {target_name} (E-value: {e_value})")

    if not top_hits:
        print("⚠️ No significant structural matches found.")

    return top_hits

### Step 2: Physicochemical & Structural Profiling

In [3]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.PDB import PDBParser
import warnings

def analyze_properties(sequence, pdb_filename):
    print("\n--- STEP 2: Physicochemical & Structural Profile ---")

    analysis = ProteinAnalysis(sequence)
    mol_weight = analysis.molecular_weight()
    pI = analysis.isoelectric_point()
    gravy = analysis.gravy()
    instability = analysis.instability_index()

    print(f"1D Properties:")
    print(f"  - Molecular Weight: {mol_weight:.2f} Da")
    print(f"  - Isoelectric Point (pI): {pI:.2f}")
    print(f"  - GRAVY (Hydrophobicity): {gravy:.3f}")
    print(f"  - Instability Index: {instability:.2f}")

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        parser = PDBParser()
        structure = parser.get_structure("Predicted", pdb_filename)

    atoms = list(structure[0].get_atoms())
    coords = [atom.get_coord() for atom in atoms]

    if coords:
        x_span = max(c[0] for c in coords) - min(c[0] for c in coords)
        y_span = max(c[1] for c in coords) - min(c[1] for c in coords)
        z_span = max(c[2] for c in coords) - min(c[2] for c in coords)
    else:
        x_span = y_span = z_span = 0.0

    print(f"3D Properties:")
    print(f"  - Total Atoms in Model: {len(atoms)}")
    print(f"  - Spatial Dimensions: {x_span:.1f}Å x {y_span:.1f}Å x {z_span:.1f}Å")

    return {"pI": pI, "gravy": gravy, "instability": instability}

### Step 3: Actionable Wet-Lab Recommendations

In [4]:
import re, warnings, requests
import numpy as np
from Bio.PDB import PDBParser, ShrakeRupley
from Bio.SeqUtils import seq1
import pydssp

MAX_ASA = dict(A=129, R=274, N=195, D=193, C=167, E=223, Q=225, G=104, H=224, I=197,
               L=201, K=236, M=224, F=240, P=159, S=155, T=172, W=285, Y=263, V=174)
HYDROPHOBIC = set("AVILMFWC")

def _runs(mask, min_len):
    """Contiguous True runs of at least min_len -> list of (start, end), 1-based."""
    out, start = [], None
    for i, m in enumerate(list(mask) + [False]):
        if m and start is None: start = i
        if not m and start is not None:
            if i - start >= min_len: out.append((start + 1, i))
            start = None
    return out

def analyze_structure(pdb_filename):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = PDBParser(QUIET=True).get_structure("m", pdb_filename)[0]
    res = [r for r in model.get_residues() if r.id[0] == " " and "CA" in r]
    seq = "".join(seq1(r.get_resname()) for r in res)
    n = len(res)

    # Confidence (pLDDT lives in the B-factor column; some tools use a 0-1 scale)
    raw = np.array([r["CA"].get_bfactor() for r in res])
    has_plddt = (not os.path.basename(pdb_filename).startswith("rcsb_")) and raw.max() > 0
    plddt = (raw * 100 if raw.max() <= 1 else raw) if has_plddt else np.full(n, np.nan)
    if plddt.max() <= 1.0: plddt = plddt * 100

    # Compactness vs. empirical expectation for globular proteins
    ca = np.array([r["CA"].get_coord() for r in res])
    rg = float(np.sqrt(((ca - ca.mean(0)) ** 2).sum(1).mean()))
    rg_exp = 2.2 * n ** 0.38

    # Relative solvent accessibility
    ShrakeRupley().compute(model, level="R")
    rsa = np.array([min(r.sasa / MAX_ASA.get(a, 200), 1.0) for r, a in zip(res, seq)])
    exposed = rsa > 0.25
    n_exp = max(exposed.sum(), 1)
    surf_hphob = sum(exposed[i] and seq[i] in HYDROPHOBIC for i in range(n)) / n_exp
    surf_charge = sum((seq[i] in "KR") - (seq[i] in "DE") for i in range(n) if exposed[i])

    # Secondary structure (pure-python DSSP)
    ss_frac = None
    try:
        coords = np.array([[r["N"].get_coord(), r["CA"].get_coord(),
                            r["C"].get_coord(), r["O"].get_coord()] for r in res])
        ss = pydssp.assign(coords, out_type="c3")
        ss = np.array(ss).astype(str)
        ss_frac = {"helix": float((ss == "H").mean()), "strand": float((ss == "E").mean())}
        ss_frac["coil"] = 1 - ss_frac["helix"] - ss_frac["strand"]
    except Exception as e:
        print(f"(secondary structure skipped: {e})")

    return dict(n=n, seq=seq, plddt=plddt, has_plddt=has_plddt, rg=rg, rg_exp=rg_exp, rsa=rsa,
            surf_hphob=surf_hphob, surf_charge=surf_charge, ss=ss_frac,
            disordered=_runs(plddt < 50, 10), low_conf=_runs(plddt < 70, 10))

def best_hit(foldseek_hits):
    hits = list(foldseek_hits.values())
    return max(hits, key=lambda h: (float(h.get("prob", 0)), -float(h.get("eval", 1e9)))) if hits else None

def transfer_uniprot_annotation(hit, plddt):
    """Map UniProt active/binding/metal sites from an AFDB hit onto the query via the alignment."""
    m = re.match(r"AF-([A-Z0-9]+)-F\d+", hit.get("target", ""))
    if not m: return None
    acc = m.group(1)
    r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=30)
    if r.status_code != 200: return None
    j = r.json()
    q, t = int(hit["qStartPos"]), int(hit["dbStartPos"])
    mapping = {}
    for qa, ta in zip(hit["qAln"], hit["dbAln"]):
        if qa != "-" and ta != "-": mapping[t] = (q, qa == ta)
        q += qa != "-"; t += ta != "-"

    name = (j.get("proteinDescription", {}).get("recommendedName", {})
              .get("fullName", {}).get("value", "n/a"))
    func = next((c["texts"][0]["value"] for c in j.get("comments", [])
                 if c.get("commentType") == "FUNCTION" and c.get("texts")), None)
    sites = []
    for f in j.get("features", []):
        if f["type"] not in ("Active site", "Binding site", "Metal binding"): continue
        s, e = f["location"]["start"]["value"], f["location"]["end"]["value"]
        mapped = [mapping[p] for p in range(s, e + 1) if p in mapping]
        if not mapped: continue
        qpos = [p for p, _ in mapped]
        sites.append(dict(type=f["type"], target_range=(s, e), query_range=(min(qpos), max(qpos)),
                          conserved=all(c for _, c in mapped),
                          ligand=f.get("ligand", {}).get("name"),
                          site_plddt=float(np.mean([plddt[p - 1] for p in qpos if p <= len(plddt)]))))
    return dict(accession=acc, name=name, function=func, sites=sites)

def analyze_tertiary_structure(pdb_filename, foldseek_hits):
    print("\n--- STEP 3: Tertiary Structure Analysis ---")
    s = analyze_structure(pdb_filename)

    if s["has_plddt"]:
        mean_pl = s["plddt"].mean()
        print(f"CONFIDENCE: mean pLDDT {mean_pl:.1f} "
              f"({'high' if mean_pl > 80 else 'moderate' if mean_pl > 70 else 'LOW - treat everything below as tentative'})")
        if s["disordered"]:
            print(f"DISORDER: likely disordered regions (pLDDT<50): {s['disordered']}")
        else:
            print("DISORDER: no extended low-pLDDT regions.")
    else:
        print("CONFIDENCE: experimental structure, so no pLDDT. Treat as high confidence "
              "(for NMR, check ensemble variability if you need flexibility).")

    # 3. Fold content and compactness
    if s["ss"]:
        print(f"SECONDARY STRUCTURE: {s['ss']['helix']:.0%} helix, {s['ss']['strand']:.0%} strand, {s['ss']['coil']:.0%} coil")
    ratio = s["rg"] / s["rg_exp"]
    print(f"COMPACTNESS: Rg {s['rg']:.1f} A vs ~{s['rg_exp']:.1f} A expected for a globular protein of {s['n']} aa "
          f"({'compact' if ratio < 1.15 else 'extended or multi-domain/disordered tails'})")

    # 4. Surface chemistry
    print(f"SURFACE: {s['surf_hphob']:.0%} of exposed residues are hydrophobic "
          f"({'unusually high - possible interaction patch, membrane association or aggregation risk' if s['surf_hphob'] > 0.35 else 'typical'}); "
          f"net exposed charge {s['surf_charge']:+d}")

    # 5. Homology and annotation transfer
    def coverage(h):
        return (int(h["qEndPos"]) - int(h["qStartPos"]) + 1) / int(h["qLen"]) if "qLen" in h else float("nan")

    best = best_hit(foldseek_hits)
    if not best:
        print("HOMOLOGY: no Foldseek hits. Candidate fold novelty, or the model is poor. "
              "Consider pocket detection (e.g. fpocket) on the PDB.")
        return s
    print(f"HOMOLOGY (best overall): {best.get('target')} | prob {float(best.get('prob', 0)):.2f} | "
          f"seq id {best.get('seqId', 'n/a')} | coverage {coverage(best):.0%} | E-value {best.get('eval', 'n/a')}")

    afdb_hit = next((h for k, h in foldseek_hits.items() if k.startswith("afdb")), None)
    if afdb_hit is None:
        print("  No AFDB hit, so no automatic annotation transfer.")
        return s

    prob, qcov = float(afdb_hit.get("prob", 0)), coverage(afdb_hit)
    print(f"  Annotation source (AFDB): {afdb_hit.get('target')} | prob {prob:.2f} | coverage {qcov:.0%}")
    if prob < 0.5 or not qcov >= 0.5:
        print("  -> Weak or partial match; shared fold is possible but don't infer function.")
        return s

    ann = transfer_uniprot_annotation(afdb_hit, s["plddt"])
    if not ann:
        print("  (Could not retrieve UniProt annotation for this hit.)")
        return s

    print(f"  Matched protein: {ann['name']} ({ann['accession']})")
    if ann["function"]:
        print(f"  Annotated function (of the HIT, not your protein): {ann['function'][:300]}")
    for site in ann["sites"]:
        flag = "CONSERVED" if site["conserved"] else "residues differ"
        pl = "" if np.isnan(site["site_plddt"]) else f", site pLDDT {site['site_plddt']:.0f}"
        print(f"  {site['type']} {site['ligand'] or ''}: hit {site['target_range']} -> "
              f"query {site['query_range']} [{flag}{pl}]")
    print("  Conserved site residues support a shared function; if they differ, "
          "the fold may be shared while the function has diverged.")
    return s

In [7]:
import os

UNIPROT_ID = "P99999"
UNKNOWN_SEQ = "MGDVEKGKKIFIMKCSQCHTVEKGGKHKTGPNLHGLFGRKTGQAPGYSYTAANKNKGIIWGEDTLMEYLENPKKYIPGTKMIFVGIKKKEERADLIAYLKKATNE"

try:
    pdb_file, pdb_data = fetch_alphafold_structure(UNIPROT_ID)

    hits = identify_structure(pdb_file, pdb_data)
    props = analyze_properties(UNKNOWN_SEQ, pdb_file)
    analyze_tertiary_structure(pdb_file, hits)

    if os.path.exists(pdb_file):
        os.remove(pdb_file)

except Exception as e:
    print(f"\nPipeline Failed: {e}")

--- STEP 1A: Fetching 3D Structure (AlphaFold DB for P99999) ---
Querying AlphaFold Database (Instant lookup)...
✅ 3D Structure downloaded instantly and saved as alphafold_P99999.pdb

--- STEP 1B: Tertiary Identification (Foldseek) ---
✅ Top match in afdb50: AF-A0A1S3FTQ6-F1-model_v6 Cytochrome c, somatic (E-value: 4.679e-21)
✅ Top match in pdb100: 5dfs-assembly2.cif.gz_B Crystal structure of Spider Monkey Cytochrome C at 1.15 Angstrom (E-value: 4.954e-21)

--- STEP 2: Physicochemical & Structural Profile ---
1D Properties:
  - Molecular Weight: 11748.60 Da
  - Isoelectric Point (pI): 9.59
  - GRAVY (Hydrophobicity): -0.706
  - Instability Index: 11.38
3D Properties:
  - Total Atoms in Model: 824
  - Spatial Dimensions: 37.4Å x 33.7Å x 34.8Å

--- STEP 3: Tertiary Structure Analysis ---
CONFIDENCE: mean pLDDT 97.9 (high)
DISORDER: no extended low-pLDDT regions.
SECONDARY STRUCTURE: 40% helix, 2% strand, 58% coil
COMPACTNESS: Rg 12.5 A vs ~12.9 A expected for a globular protein of 105 aa